<hr style="height: 6px; border: none; background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%);">
<h1 style="font-size: 48px; margin: 0; text-align: left;"> Get Data from Empatica `.avro` Files </h1>
<hr style="height: 6px; border: none; background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%);">


Reference Documentation: https://avro.apache.org/docs/1.11.1/getting-started-python/


In [1]:
import numpy  as np
import pandas as pd

import json, csv, os, sys

from datetime import datetime, timezone
from zoneinfo import ZoneInfo

### `avro`-related Packages
`%pip install avro`

In [2]:
import avro
from avro.datafile import DataFileReader
from avro.io import DatumReader

### Project Code

In [3]:
sys.path.append(os.path.abspath("../.."))

from src.utils.avro_utils.empatica_avro import get_list_of_files_to_convert, empatica_sensor_avro_to_csv
from src.utils.avro_utils.timestamps    import get_start_end_stamps, get_file_timestamp
from src.utils.logging.logging          import RESET, BOLD, UNBOLD, BLUE
from src.utils.logging.file_info        import print_directory_info

---
# Config / Path Setup

In [4]:
# Date of recording
day = "2025-10-06" # "2024-10-24" "2025-01-05"

# Which data we are looking for (see next section for more)
session_number = 1

In [5]:
# Path to avro files
INPUT_PATH = "../../../../therabot-old/lab_data/notebooks/test_data/test_1/all_3"

# Setup csv output directory
OUTPUT_DIR = "../data/test_1"
if not os.path.exists(OUTPUT_DIR): os.makedirs(OUTPUT_DIR)

# Preview input data
print_directory_info(INPUT_PATH)

../../../../therabot-old/lab_data/notebooks/test_data/test_1/all_3   76.3 MB
 ├── 1-1-15_1759709288.avro    1.0 MB
 ├── 1-1-15_1759711093.avro  974.9 KB
 ├── 1-1-15_1759712902.avro  818.9 KB
 ├── 1-1-15_1759718918.avro  867.1 KB
 ├── 1-1-15_1759720725.avro  921.7 KB
 ├── 1-1-15_1759722529.avro  919.7 KB
 ├── 1-1-15_1759724339.avro  777.7 KB
 ├── 1-1-15_1759726148.avro  774.9 KB
 ├── 1-1-15_1759727950.avro  774.6 KB
 ├── 1-1-15_1759729756.avro  774.3 KB
 ├── 1-1-15_1759731564.avro  762.0 KB
 ├── 1-1-15_1759733371.avro  880.6 KB
 ├── 1-1-15_1759735176.avro  907.0 KB
 ├── 1-1-15_1759736985.avro  851.7 KB
 ├── 1-1-15_1759738785.avro  886.7 KB
 ├── 1-1-15_1759740591.avro  838.4 KB
 ├── 1-1-15_1759742398.avro  931.0 KB
 ├── 1-1-15_1759744203.avro  751.8 KB
 ├── 1-1-15_1759746010.avro  741.6 KB
 ├── 1-1-15_1759747814.avro  749.0 KB
 ├── 1-1-15_1759749623.avro  751.5 KB
 ├── 1-1-15_1759751430.avro  767.6 KB
 ├── 1-1-15_1759753232.avro  901.2 KB
 ├── 1-1-15_1759755038.avro    1.0 MB
 ├── 1-1-15

---
# Get `start` and `end` timestamps for the recording period

Manual `start` and `end` times for different recording sessions (add a new buffer for your specific data/sessions). Add short buffers to the start and end of the actual recording start/end times to make sure you get everything. 

In [6]:
# Thesis videos
start_end_buffer = {
    1: {"start": "11:42:34.000", "end": "12:25:05.000"},
    2: {"start": "13:42:12.000", "end": "14:07:26.000"}, 
    3: {"start": "16:16:16.000", "end": "16:59:53.000"},
    4: {"start": "18:34:47.000", "end": "18:52:19.223"},
    5: {"start": "18:53:29.000", "end": "19:08:42.808"},
    
    6: {"start": "12:44:21.000", "end": "13:33:29.000"},
    7: {"start": "18:19:24.000", "end": "18:59:07.000"},
    8: {"start": "09:59:36.000", "end": "10:38:13.000"},
}

# Autism study data for each participant
start_end_buffer = {
    1: {"start": "14:57:00.000", "end": "15:16:00.000"}, # participant 4
    2: {"start": "16:15:00.000", "end": "16:28:00.000"}, # participant 3, 3 min buffer
    3: {"start": "15:05:00.000", "end": "16:17:00.000"}, # participant 3, 30 min buffer
}

### Use the `session_number` variable to change which sessions files to look for

In [7]:
# Videos were recorded in the Chicago timezone
chicago_tz = ZoneInfo("America/Chicago")

# Use the helper function to convert the manual/string format times into real timestamp objects
start_timestamp, end_timestamp = get_start_end_stamps(day, session_number, start_end_buffer, chicago_tz)

Start: 1759780620  2025-10-06 14:57:00-05:00
End:   1759781760  2025-10-06 15:16:00-05:00


---
# Select only the needed `.avro` files using the date + start/end timestamps

In [8]:
use_timestamps, avro_dict = get_list_of_files_to_convert(INPUT_PATH, start_timestamp, end_timestamp, chicago_tz)

Originally 91 .avro files
Now only converting 1 files: [1759778481]

.avro file names being used:
  1-1-15_1759778481.avro - (data starts at: 2025-10-06 14:21:21-05:00)


### Optionally, just enter the list of file names manually...

In [9]:
MANUAL_FILE_SELECT = False

In [10]:
# --------------------------------------------------------------------------------
# Print a list of the files and what timestamp they are for
# --------------------------------------------------------------------------------
if MANUAL_FILE_SELECT:

    raw_files = os.listdir(INPUT_PATH)
    raw_files.sort()

    # The files have the timestamp in their name
    for raw in raw_files:
        ts = raw.split(".")[0].split("_")[-1]
        dt = datetime.fromtimestamp(int(ts), tz=chicago_tz)
        #dt = dt - pd.Timedelta(hours=5)
        dt = dt.strftime("%Y-%m-%d %I:%M:%S %p %z")

        print(f"{ts}  {dt}")

In [11]:
# --------------------------------------------------------------------------------
# Copy and paste file names from the above cells output into this list
# --------------------------------------------------------------------------------
if MANUAL_FILE_SELECT:
    # List filenames here
    manual_avro_filenames = [
        '1-1-AMTEST01_1759786023.avro', '1-1-AMTEST01_1759787830.avro', 
        '1-1-AMTEST01_1759789636.avro', '1-1-AMTEST01_1759791443.avro', 
        '1-1-AMTEST01_1759793249.avro', '1-1-AMTEST01_1759795056.avro'
    ]
    manual_avro_filenames = raw_files
    
    # Create the same objects that the above function would
    use_timestamps, avro_dict = [], {}
    for filename in manual_avro_filenames:
        # Get the timestamp on the file
        file_timestamp = get_file_timestamp(filename)
        
        # Add to the objects
        use_timestamps.append(file_timestamp)
        avro_dict[file_timestamp] = filename


### Timestamps of files to use

In [12]:
print(use_timestamps)

[1759778481]


---
# Convert each of the selected `.avro` files
`append_mode` changes it so that if you have 3 files for a session for example, it will either output 3 `.csv` files (one for each `.avro` used), or it will append to the first file and you will get a single `.csv` output.

In [13]:
# All sensor types
empatica_sensors = ["accelerometer", "gyroscope", "temperature", "eda", "steps", "bvp"]

# Do each avro file
append_mode = False
for ts in use_timestamps:
    # Get full path to the avro file
    avro_file_path = f"{INPUT_PATH}/{avro_dict[ts]}"
    print(f"{avro_dict[ts]}")
    print("    sensor          hz   len_data   timestampStart   ")
    print("    --------------- ---- ---------  -----------------")
    
    # Read Avro file
    reader = DataFileReader(open(avro_file_path, "rb"), DatumReader())
    schema = json.loads(reader.meta.get('avro.schema').decode('utf-8'))
    data   = next(reader)
    
    # Do each of the sensors (data will be in a separate file for each sensor)
    for sensor in empatica_sensors:
        filename = f"{sensor}.csv"
        empatica_sensor_avro_to_csv(
            sensor, data, f"{OUTPUT_DIR}", 
            filename=filename, append_mode=append_mode, verbose=True,
        )
        
    # After the first file, switch to append mode
    append_mode = True
    print()

1-1-15_1759778481.avro
    sensor          hz   len_data   timestampStart   
    --------------- ---- ---------  -----------------
    accelerometer   64.0   115,264  1759778481.117390
    gyroscope        0.0         0           0.000000
    temperature      1.0     1,801  1759778482.115376
    eda              4.0     7,204  1759778481.241078
    steps            0.2       360  1759778485.869861
    bvp             64.0   115,264  1759778480.866413



### Check created files

In [14]:
# Show the output directory
print_directory_info(OUTPUT_DIR)

../data/test_1   11.6 MB
 ├── accelerometer.csv    6.7 MB
 ├── bvp.csv              4.6 MB
 ├── eda.csv            275.9 KB
 ├── gyroscope.csv       34.0  B
 ├── steps.csv            7.2 KB
 └── temperature.csv     50.4 KB


---
# Inspect one file
`bvp` functionally equivalent to `PPG`...

In [15]:
# Load the BVP data
df = pd.read_csv(f"{OUTPUT_DIR}/bvp.csv")

# Adjust some formatting
df = df.rename(columns={"unix_timestamp": "ts"})
df = df.sort_values(by="ts", ascending=True)
df["ts"] = df["ts"] / 1e6
df["|"] = "|"

# Adjust the date time column
df["dt"] = df["ts"].apply(lambda x: datetime.fromtimestamp(x, tz=chicago_tz)) 
df["d"] = df["ts"] - df["ts"].shift(1)

# Check for skips greater than one normal epoch (can happen if appending multiple avro files)
print(df["d"].value_counts())

display(df.head())
display(df.tail())

d
0.015625    115263
Name: count, dtype: int64


,ts,bvp,|,dt,d
0,1.759778e+09,0.008281,|,2025-10-06 14:21:20.866413-05:00,NaN
1,1.759778e+09,0.003111,|,2025-10-06 14:21:20.882038-05:00,0.015625
2,1.759778e+09,-0.000665,|,2025-10-06 14:21:20.897663-05:00,0.015625
3,1.759778e+09,-0.002491,|,2025-10-06 14:21:20.913288-05:00,0.015625
4,1.759778e+09,-0.002044,|,2025-10-06 14:21:20.928913-05:00,0.015625


,ts,bvp,|,dt,d
115259,1.759780e+09,0.000218,|,2025-10-06 14:51:21.788288-05:00,0.015625
115260,1.759780e+09,-0.000112,|,2025-10-06 14:51:21.803913-05:00,0.015625
115261,1.759780e+09,-0.000404,|,2025-10-06 14:51:21.819538-05:00,0.015625
115262,1.759780e+09,-0.000661,|,2025-10-06 14:51:21.835163-05:00,0.015625
115263,1.759780e+09,-0.000880,|,2025-10-06 14:51:21.850788-05:00,0.015625
